In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
# 파일 불러오기
subway = pd.read_csv('subway_with_gu.csv')
bike = pd.read_csv('bike_with_gu.csv')

In [9]:
# 날짜 포맷 통일: YYYYMMDD → datetime
bike['대여일시'] = pd.to_datetime(bike['대여일시'])
subway['사용일자'] = pd.to_datetime(subway['사용일자'].astype(str), format='%Y%m%d')

In [10]:
# 컬럼명 통일: 날짜 → '일자', 사용량 → '총사용수'
bike = bike.rename(columns={'대여일시': '일자', '총사용수': '총사용수'})
subway = subway.rename(columns={'사용일자': '일자', '승차총승객수': '총사용수'})

In [11]:
# 공통된 컬럼: 일자, 자치구, 총사용수
bike['type'] = 'Bike'
subway['type'] = 'Subway'

In [12]:
# 병합
merged_usage = pd.concat([bike[['일자', '자치구', '총사용수', 'type']], subway[['일자', '자치구', '총사용수', 'type']]])

In [13]:
merged_usage.head()

,일자,자치구,총사용수,type
0,2024-01-01,강남구,1319,Bike
1,2024-01-01,강동구,1855,Bike
2,2024-01-01,강북구,731,Bike
3,2024-01-01,강서구,5668,Bike
4,2024-01-01,관악구,1438,Bike


In [14]:
# 결과 저장
merged_usage.to_csv("final_merged.csv", index=False)

In [15]:
subway.shape

(8784, 4)

In [16]:
bike.shape

(9150, 4)

In [4]:
# 피벗 테이블 생성
pivot_df = merged_usage.pivot_table(index=['일자', '자치구'],
                          columns='type',
                          values='총사용수',
                          aggfunc='sum').reset_index()

# 컬럼 이름 정리
pivot_df.columns.name = None  # 'type' 제거
pivot_df.rename(columns={'Bike': '총사용수(B)', 'Subway': '총사용수(S)'}, inplace=True)

# 결측치 0으로 채우고 정수형 변환
pivot_df['총사용수(B)'] = pivot_df['총사용수(B)'].fillna(0).astype(int)
pivot_df['총사용수(S)'] = pivot_df['총사용수(S)'].fillna(0).astype(int)

# 저장 (선택)
pivot_df.to_csv('final_transformed.csv', index=False)